In [1]:
import json
import pandas as pd
import os
print(os.getcwd())
# print(os.path.abspath(base_path))


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

/Users/manilesrun/Desktop/Queryanalyser/QA/results_mmlu


In [2]:
base_path = "../results_mmlu"

folders = {

    3: "test_eval_service_3",
    5: "test_eval_service_5",
    8: "test_eval_service_8",
    10: "test_eval_service_10",
    12: "test_eval_service_12",
    14: "test_eval_service_14"

}

In [3]:
data = {}

for k, folder_name in folders.items():

    folder_path = os.path.join(base_path, folder_name)
#     print(folder_path)

    data[k] = []

    for file_name in os.listdir(folder_path):

        if not file_name.endswith(".json"):
            continue

        if "_efficiency" in file_name:
            continue

        file_path = os.path.join(folder_path, file_name)
#         print(file_path)

        with open(file_path, "r") as f:
            rows = json.load(f)
#             print(rows)

        for row in rows:
            row = row.copy()
            row["source_file"] = file_name
            data[k].append(row)

    print(f"K={k}: loaded {len(data[k])} rows")

K=3: loaded 7200 rows
K=5: loaded 10800 rows
K=8: loaded 16200 rows
K=10: loaded 19800 rows
K=12: loaded 23400 rows
K=14: loaded 27000 rows


In [4]:
df_k3 = pd.DataFrame(data[3])
df_k5 = pd.DataFrame(data[5])
df_k8 = pd.DataFrame(data[8])
df_k10 = pd.DataFrame(data[10])
df_k12 = pd.DataFrame(data[12])
df_k14 = pd.DataFrame(data[14])


print("K=3:", df_k3.shape)
print("K=5:", df_k5.shape)
print("K=8:", df_k8.shape)
print("K=10:", df_k10.shape)
print("K=12:", df_k12.shape)
print("K=14:", df_k14.shape)

K=3: (7200, 43)
K=5: (10800, 43)
K=8: (16200, 43)
K=10: (19800, 43)
K=12: (23400, 43)
K=14: (27000, 43)


In [5]:
df_k3["model_name"].unique()

array(['Qwen/Qwen3-Embedding-0.6B', 'mixedbread-ai/mxbai-rerank-large-v2',
       'Qwen/Qwen3-Reranker-0.6B', 'mixedbread-ai/mxbai-colbert-large-v1',
       'Qwen/Qwen3-Embedding-4B',
       'sentence-transformers/all-MiniLM-L6-v2', 'colbert-ir/colbertv2.0',
       'phi4:14b', 'Qwen/Qwen3-Reranker-4B', 'Qwen/Qwen3-Reranker-8B',
       'qwen2.5:32b', 'BAAI/bge-reranker-v2-m3', 'qwen2.5:14b',
       'lightonai/Reason-ModernColBERT', 'Qwen/Qwen3-Embedding-8B',
       'llama3.1:8b', 'BAAI/bge-large-en-v1.5',
       'LiquidAI/LFM2-ColBERT-350M'], dtype=object)

In [20]:
MODEL = "qwen2.5:14b"

In [21]:
d3 = df_k3[df_k3["model_name"] == MODEL].copy()

d5 = df_k5[df_k5["model_name"] == MODEL].copy()

d8 = df_k8[df_k8["model_name"] == MODEL].copy()

d10 = df_k10[df_k10["model_name"] == MODEL].copy()
d12 = df_k12[df_k12["model_name"] == MODEL].copy()
d14 = df_k14[df_k14["model_name"] == MODEL].copy()


print("K=3", d3.shape)
print("K=5", d5.shape)
print("K=8", d8.shape)

print("K=10", d10.shape)
print("K=12", d12.shape)
print("K=14", d14.shape)


K=3 (400, 43)
K=5 (600, 43)
K=8 (900, 43)
K=10 (1100, 43)
K=12 (1300, 43)
K=14 (1500, 43)


In [22]:
def recall_summary(df):

    specialist_df = df[
        df["expected_service"] != "general"
    ]

    general_df = df[
        df["expected_service"] == "general"
    ]

    accuracy = df["is_correct"].mean()

    specialist_recall = (
        specialist_df["predicted_label"]
        == specialist_df["expected_service"]
    ).mean()

    general_recall = (
        general_df["predicted_label"]
        == "general"
    ).mean()

    print(
        f"Accuracy: {accuracy:.4f}"
    )

    print(
        f"Specialist Recall: "
        f"{specialist_recall:.4f}"
    )

    print(
        f"General Recall: "
        f"{general_recall:.4f}"
    )


In [23]:
recall_summary(d3)
recall_summary(d5)
recall_summary(d8)
recall_summary(d10)
recall_summary(d12)
recall_summary(d14)


Accuracy: 0.9625
Specialist Recall: 0.9533
General Recall: 0.9900
Accuracy: 0.9267
Specialist Recall: 0.9300
General Recall: 0.9100
Accuracy: 0.8844
Specialist Recall: 0.9175
General Recall: 0.6200
Accuracy: 0.8782
Specialist Recall: 0.8950
General Recall: 0.7100
Accuracy: 0.8077
Specialist Recall: 0.8258
General Recall: 0.5900
Accuracy: 0.8020
Specialist Recall: 0.8150
General Recall: 0.6200


In [10]:
recall_summary(d3)

Accuracy: 0.5750
Specialist Recall: 0.4467
General Recall: 0.9600


In [11]:
cm_k3 = pd.crosstab(
    d3["expected_service"],
    d3["predicted_label"]
)

cm_k3

predicted_label,UNMATCHED,abstract_algebra,anatomy,astronomy,general
expected_service,,,,,
abstract_algebra,59,41,0,0,0
anatomy,26,0,35,0,39
astronomy,28,0,0,58,14
general,4,0,0,0,96


In [16]:
error_k3 = d3[ (d3["expected_service"] == "general") & (d3["predicted_label"] == "UNMATCHED")]

# error_k3.columns

In [17]:
error_k3[
    [
        "query",
        "expected_service",
        "predicted_label",
        "raw_response"
    ]
]

,query,expected_service,predicted_label,raw_response
3108,Can I add money with Apple Pay?,general,UNMATCHED,"The query ""Can I add money with Apple Pay?"" does not relate to any of the specialist services described. It pertains to a financial transaction method rather than abstract algebra, human anatomy, or astronomy. Therefore, the appropriate response is:\n\nservice -1"
3159,what is the amount of blood needed to stay alive,general,UNMATCHED,"The query ""what is the amount of blood needed to stay alive"" relates to physiological and cardiovascular considerations, which are connected to the study of anatomy, particularly the cardiovascular system. Among the provided specialist services, the ""anatomy"" service is the most relevant, as it involves cardiovascular auscultation and other related physiological aspects.\n\nTherefore, the query should be handled by:\n\nservice anatomy"
3194,"Hi, I have been overcharged for my payment last Saturday. I guess exchange rate was wrong.",general,UNMATCHED,"The query involves a financial issue related to overcharging and exchange rate discrepancies. None of the listed specialist services—abstract_algebra, anatomy, or astronomy—address financial transactions, currency exchange, or overcharging issues.\n\nTherefore, the appropriate response is:\n\nservice -1"
3196,is it okay to use butter instead of margarine,general,UNMATCHED,"The query ""is it okay to use butter instead of margarine"" does not match the core intents of any of the listed specialist services. Abstract algebra, anatomy, and astronomy all focus on specific academic and scientific domains that do not relate to culinary substitutions or food preparation advice.\n\n- Abstract algebra deals with mathematical structures and properties.\n- Anatomy focuses on biological structures and physiological functions.\n- Astronomy covers celestial bodies and their dynamics.\n\nThe query does not align with any of these areas, so it does not fit within the scope of the provided specialist services.\n\nTherefore, the appropriate response is:\n\nservice -1"


# K5

In [72]:
recall_summary(d5)

Accuracy: 0.8633
Specialist Recall: 0.8660
General Recall: 0.8500


In [73]:
cm_k5 = pd.crosstab(
    d5["expected_service"],
    d5["predicted_label"]
)
cm_k5

predicted_label,abstract_algebra,anatomy,astronomy,business_ethics,college_computer_science,general
expected_service,,,,,,
abstract_algebra,94,0,0,0,0,6
anatomy,0,92,0,0,0,8
astronomy,0,0,91,0,0,9
business_ethics,0,0,0,87,0,13
college_computer_science,1,0,0,0,69,30
general,2,5,2,3,3,85


In [74]:
error_k5 = d5[ (d5["expected_service"] == "general") & (d5["predicted_label"] == "anatomy")]

In [75]:
error_k5[
    [
        "query",
        "expected_service",
        "predicted_label",

        "abstract_algebra_zscore",
        "anatomy_zscore",
        "astronomy_zscore",
        "business_ethics_zscore",
        "college_computer_science_zscore",

#         "top_zscore",
#         "second_top_zscore",
        "zscore_margin",

        "std_score",

        "abstract_algebra_score",
        "anatomy_score",
        "astronomy_score",
        "business_ethics_score",
        "college_computer_science_score",
    ]
]

,query,expected_service,predicted_label,abstract_algebra_zscore,anatomy_zscore,astronomy_zscore,business_ethics_zscore,college_computer_science_zscore,zscore_margin,std_score,abstract_algebra_score,anatomy_score,astronomy_score,business_ethics_score,college_computer_science_score
510,"I am traveling to Germany, Will I be able to use my card there?",general,anatomy,-0.328897,1.863748,0.109632,-0.986690,-0.657794,1.7541,0.5701,-13.2500,-12.00000,-13.0000,-13.6250,-13.4375
511,"If I feel someone has my card information, can I get a new card?",general,anatomy,-0.658869,1.811890,-1.011835,-0.423559,0.282372,1.5295,0.5312,-12.8750,-11.56250,-13.0625,-12.7500,-12.3750
557,can i redeem my points by transferring them to someone as a gift,general,anatomy,0.110770,1.772316,-0.443079,-0.166155,-1.273852,1.6615,0.2257,-11.6875,-11.31250,-11.8125,-11.7500,-12.0000
568,say again please,general,anatomy,-0.926341,1.808918,-0.926341,0.094822,-0.051058,1.7141,0.8569,-10.0000,-7.65625,-10.0000,-9.1250,-9.2500
596,is it okay to use butter instead of margarine,general,anatomy,-0.321690,1.746319,0.367646,-1.125916,-0.666359,1.3787,0.5440,-12.6250,-11.50000,-12.2500,-13.0625,-12.8125


# K8

In [76]:
recall_summary(d8)

Accuracy: 0.8344
Specialist Recall: 0.8325
General Recall: 0.8500


In [77]:
cm_k8 = pd.crosstab(
    d8["expected_service"],
    d8["predicted_label"]
)

cm_k8

predicted_label,abstract_algebra,anatomy,astronomy,business_ethics,college_computer_science,general,high_school_chemistry,high_school_geography,jurisprudence
expected_service,,,,,,,,,
abstract_algebra,96,0,0,0,0,4,0,0,0
anatomy,0,95,0,0,0,5,0,0,0
astronomy,0,0,94,0,0,6,0,0,0
business_ethics,0,0,0,64,0,34,0,0,2
college_computer_science,3,0,0,0,80,17,0,0,0
general,1,1,3,0,2,85,0,0,8
high_school_chemistry,0,0,0,0,0,8,92,0,0
high_school_geography,1,0,2,2,0,35,0,52,8
jurisprudence,0,0,0,0,0,7,0,0,93


In [45]:
error_k8 = d8[ (d8["expected_service"] == "general") & (d8["predicted_label"] == "anatomy")]

In [46]:
error_k8[
    [
        "query",
        "expected_service",
        "predicted_label",

        "abstract_algebra_zscore",
        "anatomy_zscore",
        "astronomy_zscore",
        "business_ethics_zscore",
        "college_computer_science_zscore",
        "high_school_chemistry_zscore",
        "high_school_geography_zscore",
        "jurisprudence_zscore",

#         "top_zscore",
#         "second_top_zscore",
        "zscore_margin",

        "std_score",

        "abstract_algebra_score",
        "anatomy_score",
        "astronomy_score",
        "business_ethics_score",
        "college_computer_science_score",
        "high_school_chemistry_score",
        "high_school_geography_score",
        "jurisprudence_score",
    ]
]

,query,expected_service,predicted_label,abstract_algebra_zscore,anatomy_zscore,astronomy_zscore,business_ethics_zscore,college_computer_science_zscore,high_school_chemistry_zscore,high_school_geography_zscore,jurisprudence_zscore,zscore_margin,std_score,abstract_algebra_score,anatomy_score,astronomy_score,business_ethics_score,college_computer_science_score,high_school_chemistry_score,high_school_geography_score,jurisprudence_score
859,what is the amount of blood needed to stay alive,general,anatomy,0.168954,2.169365,0.709605,-0.804219,-0.317633,0.006758,-1.236741,-0.696089,1.4598,1.156,-11.6875,-9.375,-11.0625,-12.8125,-12.25,-11.875,-13.3125,-12.6875


# K10

In [47]:
recall_summary(d10)

Accuracy: 0.8091
Specialist Recall: 0.8060
General Recall: 0.8400


In [48]:
cm_k10 = pd.crosstab(
    d10["expected_service"],
    d10["predicted_label"]
)

cm_k10

predicted_label,abstract_algebra,anatomy,astronomy,business_ethics,college_computer_science,formal_logic,general,high_school_chemistry,high_school_geography,jurisprudence,world_religions
expected_service,,,,,,,,,,,
abstract_algebra,90,0,0,0,0,1,9,0,0,0,0
anatomy,0,96,0,0,0,0,3,1,0,0,0
astronomy,0,0,95,0,0,0,5,0,0,0,0
business_ethics,0,0,0,65,0,1,30,0,0,3,1
college_computer_science,3,0,0,0,70,2,25,0,0,0,0
formal_logic,0,0,0,0,0,93,7,0,0,0,0
general,0,2,3,0,3,0,84,0,0,7,1
high_school_chemistry,0,1,0,0,0,0,8,91,0,0,0
high_school_geography,1,0,1,3,0,1,40,0,41,4,9


In [49]:
error_k10 = d10[ (d10["expected_service"] == "general") & (d10["predicted_label"] == "high_school_chemistry")]

In [19]:
error_k10[
    [
        "query",
        "expected_service",
        "predicted_label"     
    ]
]

NameError: name 'error_k10' is not defined